# AAC Eval Dataset Annotation

This notebook annotates `annotation/eval_filtered.parquet`, adding the following columns:

- `split`: `clear` / `vague` / `both` / `none`
- `time_of_day`: `morning` / `afternoon` / `evening` / `night`
- `event_time`: concrete time as HH:MM
- `caregiver_clear`: specific caregiver sentence
- `caregiver_vague`: short implicit fragment
- `schedule`: list of calendar events (if applicable)

Run on the SLURM cluster via papermill: `sbatch run_annotate.sh`. All paths are relative to `PROJECT_ROOT`, read from an env var or auto-derived from the file location.

## 1. Configuration

In [1]:
import os
from pathlib import Path

# Project root: use NB_PROJECT_ROOT from env, otherwise walk two levels up from this file.
_nb_file = globals().get("__file__") or ""
PROJECT_ROOT = Path(
    os.environ.get("NB_PROJECT_ROOT") or
    (Path(_nb_file).resolve().parent.parent.parent if _nb_file else Path(".").resolve().parents[2])
)
ANNOTATION_DIR = PROJECT_ROOT / "annotation"

# Model
MODEL_ID          = os.environ.get("NB_MODEL_ID", "Qwen/Qwen2.5-7B-Instruct")
LOAD_IN_4BIT      = os.environ.get("NB_LOAD_IN_4BIT", "1") == "1"
MAX_NEW_TOKENS    = int(os.environ.get("NB_MAX_NEW_TOKENS", "256"))
MAX_PROMPT_LENGTH = int(os.environ.get("NB_MAX_PROMPT_LENGTH", "2048"))

# Output paths
INPUT_PATH     = ANNOTATION_DIR / "eval_filtered.parquet"
ANNOTATED_PATH = ANNOTATION_DIR / "eval_annotated.parquet"
LOG_PATH       = ANNOTATION_DIR / "annotation_log.jsonl"

# Batching and retry
BATCH_SIZE             = int(os.environ.get("NB_BATCH_SIZE", "8"))
MAX_ROW_RETRIES        = int(os.environ.get("NB_MAX_ROW_RETRIES", "3"))
MAX_ANNOTATION_RETRIES = int(os.environ.get("NB_MAX_ANNOTATION_RETRIES", "2"))
BACKUP_EVERY_N_RECORDS = int(os.environ.get("NB_BACKUP_EVERY_N", "10"))

# HuggingFace token (optional for Qwen)
HF_TOKEN = os.environ.get("HF_TOKEN", "")
if not HF_TOKEN:
    try:
        from dotenv import load_dotenv
        env_path = PROJECT_ROOT / "app" / ".env"
        if env_path.exists():
            load_dotenv(env_path)
            HF_TOKEN = os.environ.get("HF_TOKEN", "")
    except Exception:
        pass

# Number of rows to annotate (0 = full dataset)
N_ROWS = int(os.environ.get("NB_N_ROWS", "0"))

print(f"PROJECT_ROOT          : {PROJECT_ROOT}  (exists={PROJECT_ROOT.exists()})")
print(f"MODEL_ID              : {MODEL_ID}")
print(f"LOAD_IN_4BIT          : {LOAD_IN_4BIT}")
print(f"INPUT_PATH            : {INPUT_PATH}  (exists={INPUT_PATH.exists()})")
print(f"ANNOTATED_PATH        : {ANNOTATED_PATH}")
print(f"LOG_PATH              : {LOG_PATH}")
print(f"BATCH_SIZE            : {BATCH_SIZE}")
print(f"BACKUP_EVERY_N_RECORDS: {BACKUP_EVERY_N_RECORDS}")
print(f"N_ROWS                : {N_ROWS if N_ROWS > 0 else 'all'}")


PROJECT_ROOT          : /scratch.hpc/lorenzo.pellegrino2/aac-mcp-agent  (exists=True)
MODEL_ID              : Qwen/Qwen2.5-7B-Instruct
LOAD_IN_4BIT          : True
INPUT_PATH            : /scratch.hpc/lorenzo.pellegrino2/aac-mcp-agent/annotation/eval_filtered.parquet  (exists=True)
ANNOTATED_PATH        : /scratch.hpc/lorenzo.pellegrino2/aac-mcp-agent/annotation/eval_annotated.parquet
LOG_PATH              : /scratch.hpc/lorenzo.pellegrino2/aac-mcp-agent/annotation/annotation_log.jsonl
BATCH_SIZE            : 4
BACKUP_EVERY_N_RECORDS: 10
N_ROWS                : all


## 2. Imports and logging

In [2]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import json, logging, random as _random, re, shutil, time
from datetime import datetime
from pathlib import Path

import pandas as pd
import torch
from transformers import (
    AutoModelForCausalLM, AutoTokenizer,
    BitsAndBytesConfig, StoppingCriteria, StoppingCriteriaList,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    handlers=[logging.StreamHandler()],
)
log = logging.getLogger("annotate")

device = "cuda" if torch.cuda.is_available() else "cpu"
log.info("device: %s", device)
if device == "cuda":
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        log.info("  gpu %d: %s  (%.1f gb vram)", i, props.name, props.total_memory / 1e9)
else:
    log.warning("no gpu found — annotation will be extremely slow.")


2026-06-06 11:54:39,176 INFO device: cuda


2026-06-06 11:54:39,198 INFO   gpu 0: NVIDIA GeForce RTX 2080 Ti  (11.4 gb vram)


## 2b. Boot diagnostics

Reads the `.jsonl` log before loading the model and prints how many rows are already annotated and how many remain.

In [3]:
def _boot_diagnostics() -> None:
    n_valid = 0
    if LOG_PATH.exists():
        with open(LOG_PATH, "r", encoding="utf-8") as fh:
            for line in fh:
                try:
                    entry = json.loads(line)
                    if entry.get("parsed", {}).get("caregiver_clear"):
                        n_valid += 1
                except Exception:
                    pass

    try:
        df_tmp = pd.read_parquet(INPUT_PATH)
        n_total = len(df_tmp.drop_duplicates(subset=["sentence"]))
        if N_ROWS > 0:
            n_total = min(n_total, N_ROWS)
    except Exception:
        n_total = "?"

    print("=" * 55)
    print(f"  Righe gia annotate (log):                {n_valid}")
    print(f"  Frasi uniche da annotare (stima):        {n_total}")
    print(f"  (Il parquet finale avra 1760 righe dopo merge)")
    if isinstance(n_total, int):
        print(f"  Remaining:                               {max(0, n_total - n_valid)}")
    print(f"  log file exists:                         {LOG_PATH.exists()}")
    print("=" * 55)

_boot_diagnostics()


  Righe gia annotate (log):                0
  Frasi uniche da annotare (stima):        1740
  (Il parquet finale avra 1760 righe dopo merge)
  Remaining:                               1740
  log file exists:                         False


## 3. Load dataset

In [4]:
df_raw = pd.read_parquet(INPUT_PATH)
if N_ROWS > 0:
    df_raw = df_raw.head(N_ROWS)

# Fix A: annotate only unique sentences (efficiency);
# propagate annotation to all 1760 rows later via merge on sentence.
df_unique = df_raw.drop_duplicates(subset=["sentence"]).reset_index(drop=True)

log.info("dataset shape (all rows): %s  unique sentences to annotate: %d",
         df_raw.shape, len(df_unique))
print(df_unique.head(3).to_string())


2026-06-06 11:54:39,580 INFO dataset shape (all rows): (1760, 2)  unique sentences to annotate: 1740


                         sentence                                                                                                                                                                                                                                                                                                                                                                                                 concepts
0   The blue train is going fast.    [{'candidate_ids': [2603, 29010, 7282, 10187, 36763, 6494, 29008, 2472, 27768, 27769], 'concept_text': 'The blue train', 'gold_id': 7282}, {'candidate_ids': [9851, 8544, 5946, 17054, 6955, 8643, 27769, 27768, 5306, 25133], 'concept_text': 'go', 'gold_id': 8544}, {'candidate_ids': [8544, 5306, 9851, 36600, 17054, 5946, 38219, 8643, 36763, 38224], 'concept_text': 'fast', 'gold_id': 5306}]
1  The music is too loud in here.                                                                                                                 

## 4. Load model and tokenizer

In [5]:
log.info("loading tokenizer for %s ...", MODEL_ID)
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN if HF_TOKEN else None,
    trust_remote_code=True,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

quant_cfg = None
if LOAD_IN_4BIT and device == "cuda":
    quant_cfg = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
    )
    log.info("4-bit nf4 quantization enabled")

log.info("loading model weights ...")
t0 = time.time()
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN if HF_TOKEN else None,
    quantization_config=quant_cfg,
    device_map="auto" if device == "cuda" else None,
    torch_dtype=torch.bfloat16 if device == "cuda" else torch.float32,
)
model.eval()
log.info("model loaded in %.1f s", time.time() - t0)

if device == "cuda":
    allocated = torch.cuda.memory_allocated() / 1e9
    reserved  = torch.cuda.memory_reserved() / 1e9
    total     = torch.cuda.get_device_properties(0).total_memory / 1e9
    log.info("vram after load: %.2f gb allocated / %.2f gb reserved / %.2f gb total",
             allocated, reserved, total)


2026-06-06 11:54:39,639 INFO loading tokenizer for Qwen/Qwen2.5-7B-Instruct ...


2026-06-06 11:55:59,974 INFO HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-7B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"


2026-06-06 11:55:59,990 INFO HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-7B-Instruct/a09a35458c702b33eeacc393d103063234e8bc28/config.json "HTTP/1.1 200 OK"


2026-06-06 11:56:00,147 INFO HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-7B-Instruct/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"


2026-06-06 11:56:00,163 INFO HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-7B-Instruct/a09a35458c702b33eeacc393d103063234e8bc28/tokenizer_config.json "HTTP/1.1 200 OK"


...

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

2026-06-06 11:57:33,913 INFO HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-7B-Instruct/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"


2026-06-06 11:57:33,938 INFO HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-7B-Instruct/a09a35458c702b33eeacc393d103063234e8bc28/generation_config.json "HTTP/1.1 200 OK"


2026-06-06 11:57:33,952 INFO model loaded in 92.8 s


2026-06-06 11:57:33,954 INFO vram after load: 5.55 gb allocated / 5.73 gb reserved / 11.36 gb total


## 5. Time sampling and prompt construction

In [6]:
# _SEMANTIC_KEYWORDS: dict[str, list[str]] = {
#     "night"    : ["sleep", "bedtime"],
#     "morning"  : ["breakfast", "sunrise"],
#     "afternoon": ["nap", "lunch"],
#     "evening"  : ["dinner", "supper"],
# }

_SLOT_INFO: dict[str, tuple[str, int, int]] = {
    "morning"    : ("morning",    5, 12),
    "afternoon"  : ("afternoon", 13, 17),
    "evening"    : ("evening",   18, 20),
    "night_early": ("night",     21, 23),
    "night_late" : ("night",      0,  4),
}
_DEFAULT_SLOTS   = ["morning", "afternoon", "evening", "night_early", "night_late"]
_DEFAULT_WEIGHTS = [0.40, 0.35, 0.15, 0.07, 0.03]
_TOD_TO_INTERNAL: dict[str, list[str]] = {
    "morning"  : ["morning"],
    "afternoon": ["afternoon"],
    "evening"  : ["evening"],
    "night"    : ["night_early", "night_late"],
}

# Keyword hard-override: only truly unambiguous activity keywords that clearly
# belong to a single time-of-day slot. Keep this list SHORT and conservative.
_HARD_OVERRIDE_KEYWORDS: dict[str, list[str]] = {
    "morning"  : ["breakfast", "goodmorning"],
    "afternoon": ["lunch"],
    "evening"  : ["dinner", "supper"],
    "night"    : ["sleep", "bedtime", "goodnight", "good night"],
}

_RANDOM_WEIGHTS = [0.40, 0.35, 0.15, 0.10]   # morning / afternoon / evening / night


def _keyword_tod(sentence: str) -> str | None:
    """Return TOD if sentence contains an unambiguous hard-override keyword, else None."""
    sl = sentence.lower()
    for tod, words in _HARD_OVERRIDE_KEYWORDS.items():
        if any(w in sl for w in words):
            return tod
    return None


# Keep _presample_tod for backward compat (used by old code paths / tests).
def _presample_tod(sentence: str) -> str:
    kw = _keyword_tod(sentence)
    return kw if kw is not None else _random.choices(
        ["morning", "afternoon", "evening", "night"],
        weights=_RANDOM_WEIGHTS
    )[0]


# def _semantic_check(sentence: str) -> str | None:
#     combined = sentence.lower()
#     for tod, words in _SEMANTIC_KEYWORDS.items():
#         for w in words:
#             if w in combined:
#                 return tod
#     return None


def _sample_event_time(time_of_day: str) -> tuple[str, str, str]:
    internal_cats = _TOD_TO_INTERNAL.get(time_of_day)
    if internal_cats is None:
        cat = _random.choices(_DEFAULT_SLOTS, weights=_DEFAULT_WEIGHTS)[0]
    elif len(internal_cats) == 1:
        cat = internal_cats[0]
    else:
        cat = _random.choice(internal_cats)
    tod, lo, hi = _SLOT_INFO[cat]
    hour   = _random.randint(lo, hi)
    minute = _random.choice([0, 15, 30, 45])
    return cat, tod, f"{hour:02d}:{minute:02d}"


SYSTEM_PROMPT = (
    "You are a JSON generator for AAC (Augmentative and Alternative "
    "Communication) activity annotation.\n"
    "Given an activity sentence, output a single JSON object.\n"
    "Output ONLY compact single-line JSON: no newlines, no indentation, "
    "no spaces after colons or commas, no prose, no markdown fences.\n"
    "Use the literal string {TIME} as a placeholder wherever the clock time belongs.\n\n"
    "DOMAIN CONTEXT:\n"
    "The AAC user is a person with a communication disability. The caregiver is an\n"
    "adult (parent, support worker, teacher) who observes the AAC user and types a\n"
    "short input to help them express what they want or describe what is happening.\n"
    "The AAC user is a child; morning and afternoon are the most common slots,\n"
    "evening is occasional, and night is rare.\n\n"
    "FIELD DEFINITIONS:\n"
    "  caregiver_clear:\n"
    "    A precise, natural sentence the caregiver types when they have full context\n"
    "    and describe the situation explicitly. Must name the activity and include {TIME}.\n"
    "    Use third-person pronouns or role nouns (he/she/they/the child/the woman/etc.).\n"
    "    Max 15 words.\n\n"
    "  caregiver_vague:\n"
    "    A short implicit fragment the caregiver types when they assume shared context,\n"
    "    speaking the way a busy parent would to someone who already knows the child's\n"
    "    routine. Must NOT name the specific activity or key objects. The agent must\n"
    "    use time and schedule tools to resolve the reference.\n"
    "    Valid styles (vary these — do NOT repeat the same pattern):\n"
    "      implicit need : 'he keeps asking for it again'\n"
    "      routine ref   : 'the usual Tuesday thing'\n"
    "      location hint : 'before we have to be there'\n"
    "      person ref    : 'when the instructor arrives'\n"
    "      temporal hint : 'right after his nap'\n"
    "      situation ref : 'I think he wants the outside one'\n"
    "    AVOID all 'X thing, same Y' constructions. Max 12 words."
)

_FEW_SHOT_EXAMPLES = [
    # morning 1
    {
        "sentence": "I am hungry and I want my breakfast.",
        "json": (
            '{"time_of_day":"morning",'
            '"caregiver_clear":"He is asking for breakfast at {TIME} this morning",'
            '"caregiver_vague":"he keeps pointing at the kitchen",'
            '"schedule":[{"title":"Breakfast routine","start_time":"{TIME}",'
            '"location":"home","description":"daily routine"}]}'
        ),
    },
    # morning 2
    {
        "sentence": "Is it time for my speech therapy?",
        "json": (
            '{"time_of_day":"morning",'
            '"caregiver_clear":"He has speech therapy at {TIME} this morning",'
            '"caregiver_vague":"he keeps looking at the door",'
            '"schedule":[{"title":"Speech therapy","start_time":"{TIME}",'
            '"location":"home","description":null}]}'
        ),
    },
    # afternoon 1
    {
        "sentence": "I want to go to the park right now.",
        "json": (
            '{"time_of_day":"afternoon",'
            '"caregiver_clear":"He wants to go to the park at {TIME} this afternoon",'
            '"caregiver_vague":"he keeps pointing at the door",'
            '"schedule":[{"title":"Park outing","start_time":"{TIME}",'
            '"location":"park","description":null}]}'
        ),
    },
    # afternoon 2
    {
        "sentence": "My legs feel very tired today.",
        "json": (
            '{"time_of_day":"afternoon",'
            '"caregiver_clear":"She is tired and needs to rest at {TIME}",'
            '"caregiver_vague":"she can barely keep up with me today",'
            '"schedule":[{"title":"Rest after physiotherapy","start_time":"{TIME}",'
            '"location":"home","description":null}]}'
        ),
    },
    # evening 1
    {
        "sentence": "The doctor has cold hands.",
        "json": (
            '{"time_of_day":"evening",'
            '"caregiver_clear":"He has his doctor appointment at {TIME} this evening",'
            '"caregiver_vague":"he started crying before we even got there",'
            '"schedule":[{"title":"Doctor appointment","start_time":"{TIME}",'
            '"location":"clinic","description":null}]}'
        ),
    },
    # evening 2
    {
        "sentence": "I am happy when we go swimming.",
        "json": (
            '{"time_of_day":"evening",'
            '"caregiver_clear":"She has her swimming lesson at {TIME} this evening",'
            '"caregiver_vague":"she grabbed her bag the moment I mentioned it",'
            '"schedule":[{"title":"Swimming lesson","start_time":"{TIME}",'
            '"location":"pool","description":null}]}'
        ),
    },
    # night 1
    {
        "sentence": "My shirt feels scratchy on my neck.",
        "json": (
            '{"time_of_day":"night",'
            '"caregiver_clear":"He needs help with his pyjamas at {TIME} tonight",'
            '"caregiver_vague":"he keeps pulling at his collar",'
            '"schedule":[{"title":"Bedtime routine","start_time":"{TIME}",'
            '"location":"home","description":"nightly routine"}]}'
        ),
    },
    # night 2
    {
        "sentence": "I am very sleepy right now.",
        "json": (
            '{"time_of_day":"night",'
            '"caregiver_clear":"He is falling asleep at {TIME} tonight",'
            '"caregiver_vague":"he keeps closing his eyes mid sentence",'
            '"schedule":[]}'
        ),
    },
]

_EXAMPLES_TEXT = "\n\n".join(
    f'Input: sentence="{ex["sentence"]}"\nOutput: {ex["json"]}'
    for ex in _FEW_SHOT_EXAMPLES
)


def _build_prompt(sentence: str, time_of_day: str | None) -> str:
    if time_of_day is not None:
        tod_instruction = (
            f"IMPORTANT: This activity is assigned to the {time_of_day}. "
            f'You MUST set time_of_day to "{time_of_day}" and generate '
            f"caregiver_clear and caregiver_vague consistent with the {time_of_day}.\n\n"
        )
    else:
        tod_instruction = (
            "Determine the most appropriate time_of_day for this activity based on its "
            "semantic content. The AAC user is a child, so morning and afternoon are "
            "most common, evening occasional, night rare. If the sentence gives no clear "
            "time signal, choose the most plausible slot given a child's daily routine.\n\n"
        )
    user_msg = tod_instruction + (
        "Generate a JSON annotation for an AAC (Augmentative and Alternative "
        "Communication) activity.\n\n"
        "Input fields:\n"
        "  sentence: simple English sentence describing the activity.\n\n"
        "Output fields (one compact JSON object):\n"
        "  time_of_day      The most appropriate time of day for this activity.\n"
        '                   Exactly one of: \"morning\", \"afternoon\", \"evening\", \"night\".\n'
        "  caregiver_clear  Specific natural sentence for a caregiver. Use third-person\n"
        "                   pronouns or role nouns (he/she/they/the child/the woman/etc.).\n"
        "                   Must include the literal placeholder {TIME} where the clock\n"
        "                   time belongs. Max 15 words.\n"
        "  caregiver_vague  Short implicit fragment. Must NOT name the specific\n"
        "                   activity or objects. Informal register. Max 12 words.\n"
        "  schedule  Calendar events list:\n"
        '    [{"title":"..","start_time":"{TIME}","location":null,"description":null}]\n'
        "  Rules for schedule:\n"
        "  - Include a schedule event for ANY activity that:\n"
        "    (a) takes place outside the home (park, school, pool, gym, farm, clinic, etc.)\n"
        "    (b) is a structured home routine (medication, therapy, exercise, bath time)\n"
        "    (c) involves another person or professional (doctor, teacher, therapist, coach)\n"
        '  - Use [] ONLY for truly spontaneous domestic moments that are NOT pre-planned:\n'
        "    unscheduled TV watching, casual snack, free play at home.\n"
        "  - DEFAULT TO ADDING AN EVENT when in doubt.\n"
        "    The caregiver's calendar reflects the child's structured daily life.\n"
        "    Empty schedule is the EXCEPTION, not the rule.\n"
        '  - Never use generic titles like "Activity" or "Event".\n'
        "    Name the specific activity from the sentence (e.g. 'Swimming lesson', 'Horse riding').\n"
        '  - start_time MUST be the literal string "{TIME}".\n\n'
        "CRITICAL SEMANTIC CONSTRAINT:\n"
        "time_of_day MUST reflect when this activity naturally happens.\n"
        "  breakfast -> morning  |  dinner -> evening  |  sleep/nap -> night\n"
        "  concert/friday night -> night  |  afternoon tea -> afternoon\n"
        "The schedule title and caregiver_clear MUST match the input activity.\n\n"
        "Examples:\n"
        f"{_EXAMPLES_TEXT}\n\n"
        f'Input: sentence="{sentence}"\n'
        "Output (compact single-line JSON):"
    )
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_msg},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


print("time sampling and prompt builder ready.")


time sampling and prompt builder ready.


## 6. JSON extraction, stopping criteria and validation

In [7]:
def _parse_time(t_str: str) -> tuple[int, int] | None:
    try:
        h, m = map(int, t_str.split(":"))
        return h, m
    except Exception:
        return None


def _hour_to_slot(hour: int) -> str:
    if  5 <= hour <= 12: return "morning"
    if 13 <= hour <= 17: return "afternoon"
    if 18 <= hour <= 20: return "evening"
    return "night"


def _fallback(evt_time: str, tod: str) -> dict:
    return {"caregiver_clear": "", "caregiver_vague": "", "time_of_day": tod,
            "event_time": evt_time, "schedule": [], "tod_selection": None}


def _extract_json(text: str) -> dict | None:
    text = re.sub(r"```(?:json)?", "", text).strip()

    def _first_balanced_json(s: str) -> dict | None:
        start = s.find("{")
        if start < 0:
            return None
        depth, in_str, esc = 0, False, False
        for i, ch in enumerate(s[start:], start):
            if esc:   esc = False; continue
            if ch == "\\" and in_str: esc = True; continue
            if ch == '"':  in_str = not in_str; continue
            if in_str:     continue
            if ch == "{": depth += 1
            elif ch == "}":
                depth -= 1
                if depth == 0:
                    try:    return json.loads(s[start: i + 1])
                    except json.JSONDecodeError: return None
        return None

    first_out = text.find("Output:")
    if first_out >= 0:
        result = _first_balanced_json(text[first_out:])
        if result is not None:
            return result
    return _first_balanced_json(text)


class _BatchStopOnSubstring(StoppingCriteria):
    def __init__(self, stop_token_seqs: list[list[int]]):
        self._seqs = [torch.tensor(s) for s in stop_token_seqs if s]
        self._done: torch.Tensor | None = None

    def __call__(self, input_ids: torch.LongTensor, scores, **kwargs) -> bool:
        bsz = input_ids.shape[0]
        if self._done is None:
            self._done = torch.zeros(bsz, dtype=torch.bool, device=input_ids.device)
        for seq in self._seqs:
            n = len(seq)
            if input_ids.shape[1] >= n:
                tail    = input_ids[:, -n:]
                matches = (tail == seq.to(input_ids.device)).all(dim=1)
                self._done |= matches
        return bool(self._done.all())


def _trim_words(text: str, max_words: int) -> str:
    words = str(text).split()
    if len(words) <= int(max_words * 1.2):
        return str(text)
    truncated = " ".join(words[:max_words])
    for punct in (".", "!", "?"):
        last = truncated.rfind(punct)
        if last > len(truncated) // 2:
            return truncated[: last + 1]
    return truncated


def _align_schedule(raw_schedule, event_time: str, time_of_day: str) -> list[dict]:
    if not isinstance(raw_schedule, list):
        return []
    aligned, seen_titles = [], set()
    for event in raw_schedule[:2]:
        if not isinstance(event, dict): continue
        title = str(event.get("title", "") or "").strip()
        if len(title) < 4: continue
        norm = title.lower()
        if norm in seen_titles: continue
        seen_titles.add(norm)
        aligned.append({"title": title, "start_time": "{TIME}",
                         "location": event.get("location"), "description": event.get("description")})
    return aligned


def _validate(raw: dict | None, event_time: str,
              predetermined_tod: str | None = None) -> tuple[dict, bool]:
    """Validate LLM output.

    If `predetermined_tod` is provided (keyword or random fallback), it takes
    precedence over whatever time_of_day the model returned.
    If `predetermined_tod` is None the model's own time_of_day is used (model
    selection path), falling back to _hour_to_slot if absent/invalid.
    """
    parsed_hms = _parse_time(event_time)
    h          = parsed_hms[0] if parsed_hms else 9

    if raw is None or not isinstance(raw, dict):
        tod = predetermined_tod if predetermined_tod is not None else _hour_to_slot(h)
        return _fallback(event_time, tod), False

    # Determine final TOD
    if predetermined_tod is not None:
        time_of_day = predetermined_tod
    else:
        model_tod = str(raw.get("time_of_day", "")).lower().strip()
        time_of_day = model_tod if model_tod in {"morning", "afternoon", "evening", "night"} \
                      else _hour_to_slot(h)

    caregiver_clear = _trim_words(str(raw.get("caregiver_clear", "")), 15)
    caregiver_vague = _trim_words(str(raw.get("caregiver_vague", "")), 12)
    if not caregiver_clear or not caregiver_vague:
        return _fallback(event_time, time_of_day), False
    if "{TIME}" not in caregiver_clear:
        caregiver_clear = caregiver_clear.rstrip(".!?,") + " at {TIME}."
    schedule = _align_schedule(raw.get("schedule", []), event_time, time_of_day)
    return {"caregiver_clear": caregiver_clear, "caregiver_vague": caregiver_vague,
            "time_of_day": time_of_day, "event_time": event_time, "schedule": schedule}, True


def _render_time(annotation: dict) -> dict:
    evt = str(annotation.get("event_time", ""))
    if not evt:
        return annotation
    out = dict(annotation)
    out["caregiver_clear"] = str(out.get("caregiver_clear", "")).replace("{TIME}", evt)
    out["schedule"] = [{**ev, "start_time": evt} if ev.get("start_time") == "{TIME}" else ev
                       for ev in (out.get("schedule") or [])]
    return out


print("json extraction, stopping criteria and validation ready.")


json extraction, stopping criteria and validation ready.


## 7. Result helpers and split assignment

In [8]:
def apply_results(orig_df: pd.DataFrame, results: dict[int, dict]) -> pd.DataFrame:
    df = orig_df.copy()
    for col in ("caregiver_clear", "caregiver_vague", "time_of_day", "event_time", "schedule", "tod_selection"):
        if col not in df.columns:
            df[col] = None
    for idx, fields in results.items():
        if idx in df.index:
            for col, val in fields.items():
                df.at[idx, col] = val
    return df


def assign_split(row: pd.Series) -> str:
    has_clear = bool(str(row.get("caregiver_clear", "")).strip())
    has_vague = bool(str(row.get("caregiver_vague", "")).strip())
    if has_clear and has_vague: return "both"
    return "clear" if has_clear else ("vague" if has_vague else "none")


print("result helpers ready.")


result helpers ready.


## 8. Resumable log

In [9]:
_logged_idxs: set[int] = set()
_records_since_backup: int = 0


def _init_logged_idxs() -> dict[int, dict]:
    global _logged_idxs

    valid_data, n_sanitised, n_skipped = {}, 0, 0
    if not LOG_PATH.exists():
        return valid_data

    with open(LOG_PATH, "r", encoding="utf-8") as fh:
        for line in fh:
            try:
                entry  = json.loads(line)
                idx    = int(entry.get("idx", -1))
                parsed = entry["parsed"]
                if not parsed.get("caregiver_clear") or "{TIME}" not in parsed["caregiver_clear"]:
                    n_skipped += 1
                    continue
                for ev in parsed.get("schedule", []):
                    if isinstance(ev, dict) and ev.get("start_time") != "{TIME}":
                        ev["start_time"] = "{TIME}"
                        n_sanitised += 1
                _logged_idxs.add(idx)
                valid_data[idx] = parsed
            except Exception:
                continue

    log.info("log loaded: %d valid annotations (%d sanitised, %d skipped).",
             len(_logged_idxs), n_sanitised, n_skipped)
    return valid_data


def log_annotation(orig_idx: int, sentence: str,
                   raw_output: str, parsed: dict) -> None:
    global _logged_idxs, _records_since_backup

    if orig_idx in _logged_idxs:
        return

    entry = {
        "ts"        : datetime.utcnow().isoformat(),
        "idx"       : orig_idx,
        "sentence"  : sentence,
        "raw_output": raw_output,
        "parsed"    : parsed,
    }
    with open(LOG_PATH, "a", encoding="utf-8") as fh:
        fh.write(json.dumps(entry, ensure_ascii=False) + "\n")
        fh.flush()

    _logged_idxs.add(orig_idx)
    _records_since_backup += 1

    if _records_since_backup >= BACKUP_EVERY_N_RECORDS:
        _records_since_backup = 0
        log.info("Checkpoint: %d record annotati totali.", len(_logged_idxs))


print("log helpers ready.")


log helpers ready.


## 9. Main annotation loop

In [10]:
VALID_TIME_OF_DAY = {"morning", "afternoon", "evening", "night"}


def annotate_dataset(df: pd.DataFrame) -> pd.DataFrame:
    """Annotate the dataset using a 3-level TOD pipeline:

    Level 1 — keyword hard-override:
        Unambiguous activity keywords (breakfast, lunch, dinner, sleep, ...)
        → TOD is set deterministically; model is still called but its
          time_of_day output is overridden.  tod_selection = "keyword"

    Level 2 — model decision:
        All other sentences → LLM prompt does NOT specify a TOD; the model
        reads the sentence and picks the most plausible slot given child-
        domain context.  tod_selection = "model"

    Level 3 — random fallback:
        If the model returns an invalid / missing time_of_day, sample
        randomly with child-appropriate weights (morning 40 %, afternoon
        35 %, evening 15 %, night 10 %).  tod_selection = "random"
    """
    global _records_since_backup
    _records_since_backup = 0

    current_valid_results = _init_logged_idxs()
    todo = [(idx, row) for idx, row in df.iterrows() if idx not in _logged_idxs]

    if not todo:
        log.info("all rows already annotated, nothing to do.")
        return apply_results(df, current_valid_results)

    batches = [todo[i: i + BATCH_SIZE] for i in range(0, len(todo), BATCH_SIZE)]
    log.info("rows to process: %d | batches: %d", len(todo), len(batches))

    _stop_token_seqs = [
        tokenizer.encode(s, add_special_tokens=False)
        for s in ["\nInput:", "\nOutput:"]
    ]

    for b_idx, batch in enumerate(batches, 1):
        idxs  = [idx for idx, _ in batch]
        rows  = [row for _, row in batch]

        # --- Level 1: keyword hard-override ---
        # None means "let the model decide" (Level 2)
        keyword_tods: list[str | None] = [
            _keyword_tod(r["sentence"]) for r in rows
        ]

        pending = list(range(len(rows)))

        for attempt in range(1, MAX_ROW_RETRIES + 1):
            if not pending:
                break

            prompts = [
                # Pass keyword TOD when determined; None → model decides
                _build_prompt(rows[pi]["sentence"], keyword_tods[pi])
                for pi in pending
            ]
            enc = tokenizer(
                prompts, return_tensors="pt", padding=True,
                truncation=True, max_length=MAX_PROMPT_LENGTH,
            ).to(model.device)

            with torch.no_grad():
                out_ids = model.generate(
                    **enc,
                    max_new_tokens=MAX_NEW_TOKENS,
                    do_sample=True,
                    temperature=0.9, # higher temperature than baseline
                    stopping_criteria=StoppingCriteriaList(
                        [_BatchStopOnSubstring(_stop_token_seqs)]
                    ),
                )

            decoded = tokenizer.batch_decode(
                out_ids[:, enc["input_ids"].shape[1]:],
                skip_special_tokens=True,
            )

            still_pending = []
            for i, pi in enumerate(pending):
                raw_json   = _extract_json(decoded[i])
                kw_tod     = keyword_tods[pi]

                # --- Determine tod_selection and final TOD ---
                if kw_tod is not None:
                    # Level 1: keyword override
                    predetermined_tod = kw_tod
                    tod_selection     = "keyword"
                else:
                    # Level 2: try to use model's own time_of_day
                    model_tod = (
                        str(raw_json.get("time_of_day", "")).lower().strip()
                        if isinstance(raw_json, dict) else ""
                    )
                    if model_tod in VALID_TIME_OF_DAY:
                        predetermined_tod = None   # _validate will use model's value
                        tod_selection     = "model"
                    else:
                        # Level 3: random fallback with child-appropriate weights
                        predetermined_tod = _random.choices(
                            ["morning", "afternoon", "evening", "night"],
                            weights=[0.40, 0.35, 0.15, 0.10]
                        )[0]
                        tod_selection     = "random"

                # Use the predetermined TOD to sample a consistent event time
                sample_tod = predetermined_tod if predetermined_tod is not None else model_tod
                _, _, evt  = _sample_event_time(sample_tod)

                validated, ok = _validate(raw_json, evt,
                                           predetermined_tod=predetermined_tod)

                if ok:
                    validated["tod_selection"] = tod_selection
                    log_annotation(
                        int(idxs[pi]), rows[pi]["sentence"],
                        decoded[i], validated,
                    )
                    current_valid_results[idxs[pi]] = validated
                else:
                    still_pending.append(pi)

            pending = still_pending
            del enc, out_ids
            torch.cuda.empty_cache()

        for pi in pending:
            log.warning("idx=%d failed after %d retries.", idxs[pi], MAX_ROW_RETRIES)

        if b_idx % 5 == 0 or b_idx == len(batches):
            log.info("batch %d/%d done — annotated so far: %d",
                     b_idx, len(batches), len(_logged_idxs))

    annotated          = apply_results(df, current_valid_results)
    annotated["split"] = annotated.apply(assign_split, axis=1)
    return annotated


# Fix A: annotate on df_unique, then merge back to 1760 rows
t_start = time.time()
for _attempt in range(1, MAX_ANNOTATION_RETRIES + 2):
    log.info("=== annotation pass %d ===", _attempt)
    df_annotated_unique = annotate_dataset(df_unique)
    n_failed = (
        df_annotated_unique["caregiver_clear"].isna().sum()
        + (df_annotated_unique["caregiver_clear"] == "").sum()
    )
    if n_failed == 0:
        log.info("done. all rows annotated successfully.")
        break
    log.warning("%d rows still missing, starting recovery pass.", n_failed)

log.info("all passes completed in %.0f s.", time.time() - t_start)


2026-06-06 11:57:34,265 INFO === annotation pass 1 ===


2026-06-06 11:57:34,338 INFO rows to process: 1740 | batches: 435


2026-06-06 11:59:06,738 INFO Checkpoint: 10 record annotati totali.


2026-06-06 12:00:09,686 INFO Checkpoint: 20 record annotati totali.


2026-06-06 12:00:09,766 INFO batch 5/435 done — annotated so far: 20


...

2026-06-06 15:49:24,781 INFO Checkpoint: 1730 record annotati totali.


2026-06-06 15:50:25,322 INFO Checkpoint: 1740 record annotati totali.


2026-06-06 15:50:25,403 INFO batch 435/435 done — annotated so far: 1740


2026-06-06 15:50:25,632 INFO done. all rows annotated successfully.


2026-06-06 15:50:25,633 INFO all passes completed in 13971 s.


## 10. Render {TIME} placeholders and save

In [11]:
def _apply_render_time(row: pd.Series) -> pd.Series:
    ann = {"event_time"     : row.get("event_time", ""),
           "caregiver_clear": row.get("caregiver_clear", ""),
           "schedule"       : row.get("schedule", [])}
    rendered = _render_time(ann)
    row = row.copy()
    row["caregiver_clear"] = rendered["caregiver_clear"]
    row["schedule"]        = rendered["schedule"]
    return row

df_annotated_unique = df_annotated_unique.apply(_apply_render_time, axis=1)
df_annotated_unique = df_annotated_unique.drop(columns=["concept_texts"], errors="ignore")

# Fix A: propagate annotation back to all 1760 rows via merge on sentence.
ann_cols = ["sentence", "caregiver_clear", "caregiver_vague",
            "time_of_day", "event_time", "schedule", "tod_selection", "split"]
df_final = df_raw.merge(
    df_annotated_unique[ann_cols],
    on="sentence",
    how="left",
)

df_final.to_parquet(ANNOTATED_PATH, index=False)
log.info("saved: %s  (%d rows — all original rows incl. duplicates)",
         ANNOTATED_PATH, len(df_final))

print(f"\nFinal shape: {df_final.shape}  (expected 1760 rows)")
print("\nsplit distribution:")
print(df_final["split"].value_counts().to_string())
print("\nsample:")
print(df_final[["sentence", "caregiver_clear", "caregiver_vague",
                "time_of_day", "split"]].head(5).to_string())


2026-06-06 15:50:26,076 INFO saved: /scratch.hpc/lorenzo.pellegrino2/aac-mcp-agent/annotation/eval_annotated.parquet  (1760 rows — all original rows incl. duplicates)



Final shape: (1760, 9)  (expected 1760 rows)

split distribution:
split
both    1760

sample:
                               sentence                                          caregiver_clear                     caregiver_vague time_of_day split
0         The blue train is going fast.  He is playing with the blue train at 06:15 this morning    he keeps asking for the blue one     morning  both
1        The music is too loud in here.    He is annoyed by the loud music at 20:15 this evening          he keeps covering his ears     evening  both
2          Look at the bubbles popping.       He is enjoying the bubbles at 17:15 this afternoon  he keeps reaching for more bubbles   afternoon  both
3   My shirt feels scratchy on my neck.          He needs help with his pyjamas at 04:15 tonight      he keeps pulling at his collar       night  both
4  The fan is spinning round and round.         He is playing with the fan at 19:30 this evening              he keeps running to it     evening  both